 ## Preparing the system

In [ ]:
!cat input_files/model_cg.itp

In [ ]:
!cat input_files/molecule_unl_cg.gro

In [ ]:
import os,shutil

#simple function to write mdp files
def write_mdp(mdp_string, mdp_name, directory):
    mdp_filename=os.path.join(directory,mdp_name)
    mdp_filehandle=open(mdp_filename,'w')
    mdp_filehandle.write(mdp_string)
    mdp_filehandle.close()

#get the path to the working directory
pwd=os.getcwd()

#path to gromacs files
input_files=os.path.join(pwd,'input_files')
output_files=os.path.join(pwd,'output_files')
if not os.path.isdir(output_files):
    os.mkdir(output_files)
shutil.copy(os.path.join(input_files,'molecule_unl_cg.gro'),os.path.join(output_files,'molecule.gro'))
shutil.copy(os.path.join(input_files,'model_cg.itp'),os.path.join(output_files,'topol.top'))
shutil.copy(os.path.join(input_files,'model_cg.itp'),os.path.join(output_files,'model_cg.itp'))
shutil.copy(os.path.join(input_files,'system_cg.top'),os.path.join(output_files,'system.top'))

shutil.copy(os.path.join(input_files,'solvent_eq.gro'),os.path.join(output_files,'solvent_eq.gro'))
shutil.copy(os.path.join(input_files,'martini_v3.0.0_solvents_v1.itp'),os.path.join(output_files,'martini_v3.0.0_solvents_v1.itp'))
shutil.copy(os.path.join(input_files,'martini_v3.0.0_TC1ch.itp'),os.path.join(output_files,'martini_v3.0.0_TC1ch.itp'))

%cd output_files

In [ ]:
!gmx editconf -f molecule.gro -o box.gro -bt dodecahedron -d 1.2

In [ ]:
!gmx solvate -cp box.gro -cs solvent_eq.gro -o solvated.gro -p topol.top

In [ ]:
em_mdp=""";minimal mdp options for energy minimization
integrator               = steep
nsteps                   = 50000
coulombtype              = pme
"""
write_mdp(em_mdp,'em.mdp',output_files)
!gmx grompp -f em.mdp -c solvated.gro -p system.top -o em.tpr -maxwarn 2
!gmx mdrun -ntmpi 4 -v -deffnm em 

 ## Equilibration of energy

In [ ]:
equil_mdp=""";equilibration mdp options
integrator               = md
nsteps                   = 1000000
dt                       = 0.002
nstenergy                = 100
rlist                    = 1.1
nstlist                  = 10
rvdw                     = 1.1
coulombtype              = pme
rcoulomb                 = 1.1
fourierspacing           = 0.13
constraints              = h-bonds
tcoupl                   = v-rescale
tc-grps                  = system
tau-t                    = 0.5
ref-t                    = 300
pcoupl                   = berendsen
ref-p                    = 1
compressibility          = 4.5e-5
tau-p                    = 1
gen-vel                  = yes
gen-temp                 = 300
cutoff-scheme            = Verlet
"""
write_mdp(equil_mdp,'equil.mdp',output_files)
!gmx grompp -f equil.mdp -p system.top -c em.gro -o equil.tpr -maxwarn 1
!gmx mdrun -ntmpi 4 -v -deffnm equil 

## Creating the $\lambda$ points

In [ ]:
run_mdp="""; we'll use the sd integrator (an accurate and efficient leap-frog stochastic dynamics integrator) with 100000 time steps (200ps)
integrator               = sd
nsteps                   = 1000000
dt                       = 0.002
nstenergy                = 1000
nstcalcenergy            = 50 ; should be a divisor of nstdhdl 
nstlog                   = 5000
; cut-offs at 1.0nm
rlist                    = 1.1
rvdw                     = 1.1
; Coulomb interactions
coulombtype              = pme
rcoulomb                 = 1.1
fourierspacing           = 0.13
; Constraints
constraints              = h-bonds
; set temperature to 300K
tc-grps                  = system
tau-t                    = 2.0
ref-t                    = 300
; set pressure to 1 bar with a thermostat that gives a correct
; thermodynamic ensemble
pcoupl                   = berendsen ;C-rescale
ref-p                    = 1.0
compressibility          = 4.5e-5
tau-p                    = 5.0

; and set the free energy parameters
free-energy              = yes
couple-moltype           = UNL
nstdhdl                  = 50 ; frequency for writing energy difference in dhdl.xvg, 0 means no ouput, should be a multiple of nstcalcenergy. 
; these 'soft-core' parameters make sure we never get overlapping
; charges as lambda goes to 0
; soft-core function
sc-power                 = 1
sc-sigma                 = 0.3
sc-alpha                 = 1.0
; we still want the molecule to interact with itself at lambda=0
couple-intramol          = no
couple-lambda1           = vdwq
couple-lambda0           = none
init-lambda-state        = {}
; These are the lambda states at which we simulate
; for separate LJ and Coulomb decoupling, use
fep-lambdas              = 0.0 0.2 0.4 0.6 0.8 0.9 1.0
"""

In [ ]:
number_of_lambdas=7
for lambda_number in range(number_of_lambdas):
    lambda_directory=os.path.join(output_files,'lambda_{:0>2}'.format(lambda_number))
    os.mkdir(lambda_directory)
    gro_file=os.path.join(output_files,'equil.gro')
    top_file=os.path.join(output_files,'topol.top')
    system_file=os.path.join(output_files,'system.top')

    model_itp_file=os.path.join(output_files,'model_cg.itp')
    m3_solvent_file=os.path.join(output_files,'martini_v3.0.0_solvents_v1.itp')
    m3_model_file=os.path.join(output_files,'martini_v3.0.0_TC1ch.itp')

    
    shutil.copy(gro_file,os.path.join(lambda_directory,'conf.gro'))
    shutil.copy(top_file,lambda_directory)
    shutil.copy(system_file,lambda_directory)

    shutil.copy(model_itp_file,lambda_directory)
    shutil.copy(m3_solvent_file,lambda_directory)
    shutil.copy(m3_model_file,lambda_directory)
    
    write_mdp(run_mdp.format(lambda_number),'grompp.mdp',lambda_directory)
    %cd $lambda_directory
    !gmx grompp -p system.top -maxwarn 1
    !gmx mdrun -ntmpi 1 -v 

In [ ]:
%cd ..

In [ ]:
!ls -F 

In [ ]:
!cat lambda_00/grompp.mdp

 ## Post-processing: extracting the free energy

In [ ]:
!head -40 lambda_00/dhdl.xvg

In [ ]:
bar_string = ''
for lambda_number in range(number_of_lambdas):
    lambda_directory=os.path.join(output_files,'lambda_{:0>2}'.format(lambda_number))
    bar_string=bar_string + lambda_directory + '/dhdl.xvg '
!gmx bar -b 100 -f $bar_string

Where the -b 100 means that the first 100 ps should be disregarded: they serve as an extra equilibration step.